## Purpose:
    This script generates shortest-path baseline routes from validation OD route
    segments using the hexagon road network graph.

## Input:
    - Hexagon road network graph:
      ./data/new_hexagraph/hexa_network_with_road.gpickle
    - Validation OD route segment files:
      ./data/simulated_processed_inputs/Validation/shortcut_route/route_split_{k}/
      where k = 8, 12, and 16

## Output:
    - Shortest-path baseline route predictions:
      ./expected_findings/code5_shortcut_prediction_route/shortcut_path_{k}.csv

## Main procedures:
    1. Load the hexagon road network graph.
    2. Read validation route segment files for each window size k.
    3. Extract start and end nodes from each OD segment.
    4. Generate the shortest path between each start-end node pair using Dijkstra's algorithm.
    5. Remove duplicated boundary nodes between consecutive segments.
    6. Merge segment-level paths into traveler-level predicted routes.
    7. Save the shortest-path baseline routes as CSV files.

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import pickle
from torch_geometric.utils import from_networkx
import os
import re

## 1. Load input graph and route segment files

In [2]:
# hexagon_data
with open("./data/new_hexagraph/hexa_network_with_road.gpickle", 'rb') as f:
    G = pickle.load(f)
data = from_networkx(G)

In [3]:
for k in range(8, 17, 4): # Process route segments generated with different window sizes: k = 8, 12, and 16.
    # Load route segment files.
    csv_dir_Va = f'./data/simulated_processed_inputs/Validation/shortcut_route/route_split_{k}/'

    def numerical_sort_key(file): # Sort file names in natural numerical order.
        return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', file)]

    route_split_list_Va = sorted(
        [f for f in os.listdir(csv_dir_Va) if f.endswith(".csv")],
        key=numerical_sort_key
    )


    route_seq_list = []
    length_list = []
    travel_name_list = []
    # Read each OD route segment and store its node sequence and traveler ID.
    for file_name in route_split_list_Va:
        tmp_r = pd.read_csv(f'./data/simulated_processed_inputs/Validation/shortcut_route/route_split_{k}/' + file_name)
        seq = np.array(tmp_r.iloc[:, 0])
        travel_name = tmp_r.iloc[0, 1]
        route_seq_list.append(seq)
        length_list.append(len(seq))
        travel_name_list.append(travel_name)

    # Extract origin and destination nodes from each route segment.
    end_nodes = [seq[-1].item() for seq in route_seq_list]
    start_nodes = [seq[0].item() for seq in route_seq_list]


    # Generate shortest paths between OD node pairs.
    # Compute the shortest path using Dijkstra's algorithm.
    path_total = []
    length_total = []

    for i in range(len(end_nodes)):
        path = nx.shortest_path(G, source=int(start_nodes[i]), target=int(end_nodes[i]), weight='weight')
        length = nx.shortest_path_length(G, source=int(start_nodes[i]), target=int(end_nodes[i]), weight='weight')
        path_total.append(path)
        if i % 50 == 0:
            print(f'{i}/{len(end_nodes)}')


    # Create a dataframe
    df = pd.DataFrame({
        'name': travel_name_list,
        'path': path_total
    })
    code_list = df['name'].unique()

    for code in code_list:
        target_idx = df[df['name'] == code].index
        paths = df.loc[target_idx, 'path'].tolist()
        
        cleaned_paths = []
        for i in range(len(paths)):
            current = paths[i]
            if i < len(paths) - 1 and current[-1] == paths[i+1][0]:
                current = current[:-1]
            cleaned_paths.append(current)
        
        for i, idx in enumerate(target_idx):
            df.at[idx, 'path'] = cleaned_paths[i]

    grouped_df = df.groupby('name')['path'].sum()

    grouped_df.to_csv(f'./expected_findings/code5_shortcut_prediction_route/shortcut_path_{k}.csv')

0/812
50/812
100/812
150/812
200/812
250/812
300/812
350/812
400/812
450/812
500/812
550/812
600/812
650/812
700/812
750/812
800/812
0/515
50/515
100/515
150/515
200/515
250/515
300/515
350/515
400/515
450/515
500/515
0/374
50/374
100/374
150/374
200/374
250/374
300/374
350/374
